**НИУ ВШЭ · программа ФТиАД · магистратура, 1-й курс**

# Домашнее задание № 1 по эконометрике

## Сравнение цен домов с учётом площади и района

Сравните цены домов двух типов с учётом площади. С помощью трёх моделей выясните, как изменится оценка разницы в ценах, если учесть район.

### Что сдать и что принести на лекцию

- **До 23:59 15 ноября** отправьте файл `.ipynb` с кодом, результатами расчётов и ответами на [aj.zhimerikina@physics.msu.ru](mailto:aj.zhimerikina@physics.msu.ru). Записывайте выводы рядом с расчётами. За каждое из шести заданий можно получить 1 балл.
- На лекции **16 ноября в 18:10** вы получите три вопроса по результатам своей модели 3. На ответы отводится 30 минут. Принесите заполненный лист с последней страницы файла «Домашнее_задание_01.pdf». Во время работы можно пользоваться **только распечатанным листом с результатами**. За ответы можно получить до 4 баллов.

### Данные и номер варианта

В данных представлены сделки купли-продажи домов с одним, двумя или тремя жилыми помещениями в пяти районах Нью-Йорка с октября 2025 по сентябрь 2026 года. Каждая строка описывает одну сделку.

Файл `nyc_homes_2025-10_2026-09.csv.gz` лежит в папке `data` рядом с ноутбуком. Данные выгружены 18 октября 2026 года с сайта Департамента финансов Нью-Йорка. Источник и правила отбора описаны в `data/source_info.json`.

Укажите свой номер варианта (от 1 до 60) в `VARIANT` и запустите первую ячейку с кодом. Она отберёт 3 000 наблюдений и выведет таблицу `target` с характеристиками дома для прогноза.


In [164]:
# При необходимости установите зависимости и перезапустите ядро:
%pip install -r requirements.txt
%pip install plotly
%pip install --upgrade "nbformat>=4.2.0"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [165]:
import plotly.express as px

In [166]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm

VARIANT = 12  # Замените None на свой номер варианта от 1 до 60.
if type(VARIANT) is not int or not 1 <= VARIANT <= 60:
    raise ValueError('Замените None на номер варианта от 1 до 60.')
data_path = Path('data') / 'nyc_homes_2025-10_2026-09.csv.gz'
if not data_path.is_file():
    raise FileNotFoundError('Папка data с файлом nyc_homes_2025-10_2026-09.csv.gz должна лежать рядом с ноутбуком.')
base = pd.read_csv(data_path)
data = base.sample(n=3000, random_state=20261019 + VARIANT).copy()
target = pd.DataFrame([{
    'area_m2': [140, 160, 180, 200, 220][(VARIANT - 1) % 5],
    'multi_family': ((VARIANT - 1) // 5) % 2,
    'borough': [2, 3, 4, 5][(VARIANT - 1) % 4],
}])
print('Вариант:', VARIANT, '| Число наблюдений:', len(data))
print('Характеристики дома для прогноза:')
display(target)
data[['price', 'area_m2', 'multi_family', 'borough']].head()


Вариант: 12 | Число наблюдений: 3000
Характеристики дома для прогноза:


,area_m2,multi_family,borough
0,160,0,5


,price,area_m2,multi_family,borough
15548,532.481,193.238323,0,3
7256,910.000,110.368812,0,4
5446,900.000,100.335283,0,4
10395,735.000,131.922317,0,5
11366,1500.000,316.056142,1,2


In [167]:
selected_columns = ['price', 'area_m2', 'multi_family', 'borough']
selected_data = data[
    selected_columns
].copy()

### Описание данных

| Колонка | Обозначение | Единицы или коды | Что означает |
| --- | --- | --- | --- |
| `price` | $P$ | тыс. долл. США | Цена всего дома |
| `area_m2` | $S$ | м² | Площадь здания |
| `multi_family` | $M$ | 0 или 1 | $M=0$ для одного жилого помещения, $M=1$ для двух или трёх |
| `borough` | Район | от 1 до 5 | Manhattan (1), Bronx (2), Brooklyn (3), Queens (4), Staten Island (5) |

### Три модели

$$
\begin{aligned}
\text{Модель 1.}\quad & \ln(P)=\beta_0+\beta_1z+\beta_2M+\varepsilon \\
\text{Модель 2.}\quad & \ln(P)=\beta_0+\beta_1z+\beta_2M+G+\varepsilon \\
\text{Модель 3.}\quad & \ln(P)=\beta_0+\beta_1z+\beta_2M+G+\delta Mz+\varepsilon \\
& G=\alpha_1D_1+\alpha_2D_2+\alpha_3D_3+\alpha_5D_5 .
\end{aligned}
$$

Модель 1 учитывает площадь и тип дома. Модель 2 добавляет район. В модели 3 наклон может различаться для двух типов домов. Коэффициенты оценивайте отдельно для каждой модели.

Во всех моделях зависимая переменная — натуральный логарифм цены $\ln(P)$ по основанию $e$. Обозначим $z=\ln(S/150)$. Базовый район — Queens (код 4). Индикатор $D_j$ равен 1 для домов в районе $j$ и 0 для остальных.

### Предпосылки модели

Считайте, что при заданных значениях признаков ошибки независимы и нормально распределены с нулевым средним и одинаковой дисперсией. Используйте эти предпосылки при проверке гипотез и построении интервалов.

Модели оценивают средний логарифм цены при заданных признаках. Под прогнозом цены в задании понимаем экспоненту прогноза логарифма. При указанных предпосылках это оценка медианы цены для домов с такими признаками.


## 1. Проверьте выборку и запишите ожидания

1. Проверьте, что в вашей выборке 3 000 наблюдений.
2. Посчитайте число сделок по районам и типам домов. В строках таблицы укажите районы, в столбцах — типы домов. Проверьте, представлены ли оба типа в каждом районе.
3. До оценки моделей запишите и обоснуйте два предположения в 3–4 предложениях. Как, по-вашему, цена связана с площадью? Как может измениться оценка разницы в ценах двух типов домов, если учесть район?

Проверка количества наблюдений в выборке

In [168]:
print('Количество записей в выборке:', selected_data.shape[0])

Количество записей в выборке: 3000


In [169]:
selected_data['log_price'] = np.log(selected_data['price'])
selected_data['log_area'] = np.log(selected_data['area_m2'])

In [170]:
selected_data.head(3)

,price,area_m2,multi_family,borough,log_price,log_area
15548,532.481,193.238323,0,3,6.277547,5.263924
7256,910.000,110.368812,0,4,6.813445,4.703828
5446,900.000,100.335283,0,4,6.802395,4.608517


In [171]:
selected_data.describe().T

,count,mean,std,min,25%,50%,75%,max
price,3000.0,1195.122107,1173.517866,10.000000,712.706250,915.000000,1275.000000,21000.000000
area_m2,3000.0,182.132818,80.532364,35.767670,124.490074,166.668054,221.573750,870.966000
multi_family,3000.0,0.479000,0.499642,0.000000,0.000000,0.000000,1.000000,1.000000
borough,3000.0,3.642000,0.929941,1.000000,3.000000,4.000000,4.000000,5.000000
log_price,3000.0,6.882492,0.587115,2.302585,6.569069,6.818924,7.150701,9.952278
log_area,3000.0,5.121001,0.404016,3.577044,4.824226,5.116004,5.400755,6.769603


Число сделок по районам и типам домов. В строках таблицы указаны районы, в столбцах — типы домов.

In [172]:
# Для таблицы подойдёт pd.crosstab с borough в строках и multi_family в столбцах.
pd.crosstab(selected_data['borough'], selected_data['multi_family'], margins=True, margins_name='Итого')


multi_family,0,1,Итого
borough,,,
1,15,14,29
2,123,212,335
3,326,521,847
4,715,544,1259
5,384,146,530
Итого,1563,1437,3000


Для каждого из районов представлены оба типа домов, но для первого типа (Manhattan) мало наблюдений

Посмотрим на связь площади с ценой дома

In [173]:
fig = px.scatter(
    selected_data, x='area_m2', y='price', opacity=0.5,
    title='Реальные продажи: цена и площадь дома',
    labels={'area_m2': 'Площадь здания, м²', 'price': 'Цена, тыс. долл. США'},
)
fig.show()

In [174]:
fig = px.scatter(
    selected_data, x='log_area', y='log_price', opacity=0.5,
    title='Логарифм цены и логарифм площади',
    labels={'log_area': 'ln(площадь)', 'log_price': 'ln(цена)'},
)
fig.show()

In [175]:
area_m2_price_corr = selected_data['price'].corr(selected_data['area_m2'])
print('Корелляция цены с площадью дома:', area_m2_price_corr.round(3))

Корелляция цены с площадью дома: 0.527


Посмотрим на зависимость цены дома от района

In [176]:
borough_values = {1: "Manhattan (1)", 2: "Bronx (2)", 3: "Brooklyn (3)", 4: "Queens (4)", 5: "Staten Island (5)"}
plot_data = selected_data.assign(
    borough_name=selected_data['borough'].map(borough_values)
)

# Нормируем группы отдельно и используем общие границы интервалов.
fig = px.histogram(
    plot_data, x='log_price', color='borough_name', barmode='overlay', opacity=0.5,
    histnorm='probability density',
    category_orders={'borough_name': borough_values},
    title='Распределение логарифма цены в двух группах',
    labels={'log_price': 'ln(цена)', 'borough_name': 'Тип дома'},
)
fig.update_traces(xbins=dict(
    start=np.floor(selected_data['log_price'].min() / 0.25) * 0.25,
    end=np.ceil(selected_data['log_price'].max() / 0.25) * 0.25, size=0.25,
))
fig.update_layout(yaxis_title='Плотность')
fig.show()

In [177]:
(
    plot_data.groupby(['borough_name'])['price']
    .agg(['mean', 'median', 'std'])
    .round(2)
    .sort_values(by='mean')
)

,mean,median,std
borough_name,,,
Staten Island (5),812.97,748.75,318.15
Bronx (2),850.30,800.00,363.02
Queens (4),998.05,948.00,432.26
Brooklyn (3),1664.53,1250.00,1452.18
Manhattan (1),7008.29,6100.00,4930.29


In [178]:
plot_data_agg = (
    plot_data.groupby(['borough_name', 'multi_family'])['price']
    .agg(['mean', 'median', 'std'])
    .unstack('multi_family')
    .round(2)
)
plot_data_agg

mean            median              std         
multi_family             0        1        0       1        0        1
borough_name                                                          
Bronx (2)           750.07   908.45   685.00   877.5   453.67   283.46
Brooklyn (3)       1618.39  1693.40   968.67  1448.0  1840.06  1145.69
Manhattan (1)      8033.75  5909.59  7300.00  4137.5  4421.16  5365.77
Queens (4)          916.19  1105.64   860.00  1035.0   402.28   446.84
Staten Island (5)   760.24   951.65   711.20   925.0   308.05   303.15

Посмотрим разницу в ценах домов в одном районе между домами с одним и несколькими жилыми помещениями

In [179]:
price_diff = pd.DataFrame()
price_diff['Разница, тыс. долл. США'] = plot_data_agg['mean'][1] - plot_data_agg['mean'][0]
price_diff['Разница, %'] = (price_diff['Разница, тыс. долл. США']) / plot_data_agg['mean'][0] * 100
price_diff.round(2)


,"Разница, тыс. долл. США","Разница, %"
borough_name,,
Bronx (2),158.38,21.12
Brooklyn (3),75.01,4.63
Manhattan (1),-2124.16,-26.44
Queens (4),189.45,20.68
Staten Island (5),191.41,25.18


**Итоги проверки выборки и ожидания до оценки моделей**

> Как, по-вашему, цена связана с площадью?

Ожидаю, что при одинаковых остальных характеристиках большая площадь будет связана с более высокой ценой, поскольку покупатель получает больше пространства. Положительная парная корреляция цены и площади, равная примерно 0,527, согласуется с этим ожиданием, хотя сама по себе не подтверждает связь при прочих равных.

> Как может измениться оценка разницы в ценах двух типов домов, если учесть район?

Ожидаю, что учёт района изменит оценку разницы в ценах двух типов домов, поскольку типы неодинаково распределены по районам, а уровни цен между районами различаются. В четырёх районах средняя цена домов с двумя-тремя помещениями выше, чем домов с одним помещением, на 4,63–25,18%, а в Manhattan ниже на 26,44%.

## 2. Оцените три модели

1. Оцените модели 1–3 по отдельности методом наименьших квадратов на одной и той же выборке из 3 000 наблюдений.
2. Для каждой модели укажите число наблюдений и R² для ln(P).
3. Сведите в таблицу оценки коэффициентов, их стандартные ошибки и 95%-е доверительные интервалы.

In [180]:
y = selected_data['log_price']
selected_data['log_area150'] = np.log(selected_data['area_m2'] / 150)

In [181]:
from IPython.display import HTML
def show_model(result):
    ci = result.conf_int()
    table = pd.DataFrame({
        'Коэффициент': result.params,
        'Ст. ошибка': result.bse,
        'p-value': result.pvalues,
        'ДИ 95%: нижняя': ci[0],
        'ДИ 95%: верхняя': ci[1],
    })
    display(table)
    display(HTML('<details><summary>Полный вывод statsmodels</summary>'
                 + result.summary().as_html() + '</details>'))

### Модель 1. Учитываем площадь и тип дома
$$\ln(P)=\beta_0+\beta_1z+\beta_2M+\varepsilon.$$

$P_i$ — цена,  $z=\ln(S/150)$. $M$ - multi_family ($M=0$ для одного жилого помещения, $M=1$ для двух или трёх)

$\beta_1$ — эластичность условной медианы цены по площади при фиксированном типе дома: увеличение площади на 1% связано примерно с изменением этого показателя цены на $\beta_1\%$.

$\beta_2$ — различие ожидаемых логарифмов цен между домами с двумя-тремя помещениями и домами с одним помещением при одинаковой площади.

In [182]:
# Включите константу во все модели. В моделях с районными индикаторами базовый район — Queens.
X_1 = sm.add_constant(selected_data[['log_area150', 'multi_family']])
res_1 = sm.OLS(y, X_1).fit()
show_model(res_1)


,Коэффициент,Ст. ошибка,p-value,ДИ 95%: нижняя,ДИ 95%: верхняя
const,6.826534,0.012829,0.000000e+00,6.801379,6.851689
log_area150,0.796030,0.025927,3.150917e-180,0.745193,0.846868
multi_family,-0.066590,0.020965,1.507170e-03,-0.107698,-0.025483


**Число наблюдений**: No. Observations:	3000

**$R^2$ для $\ln P$**: 0.273

### Модель 2. Добавляем район

$$
\begin{aligned}
& \ln(P)=\beta_0+\beta_1z+\beta_2M+G+\varepsilon \\
& G=\alpha_1D_1+\alpha_2D_2+\alpha_3D_3+\alpha_5D_5.
\end{aligned}
$$

Базовый район — Queens (код 4). Индикатор $D_j$ равен 1 для домов в районе $j$ и 0 для остальных.

In [183]:
borough_dummies = pd.get_dummies(
    selected_data['borough'],
    prefix='borough',
    dtype=float,
)

# Queens (код 4) — базовый район.
borough_dummies = borough_dummies.drop(columns='borough_4')
dummy_columns = borough_dummies.columns.tolist()
selected_data[dummy_columns] = borough_dummies
dummy_columns

['borough_1', 'borough_2', 'borough_3', 'borough_5']

In [184]:
# Площадь, тип дома и районные индикаторы.
X_2 = sm.add_constant(selected_data[['log_area150', 'multi_family', *dummy_columns]])

res_2 = sm.OLS(y, X_2).fit()
show_model(res_2)

,Коэффициент,Ст. ошибка,p-value,ДИ 95%: нижняя,ДИ 95%: верхняя
const,6.840392,0.015455,0.000000e+00,6.810087,6.870696
log_area150,0.701610,0.024951,1.383506e-154,0.652688,0.750532
multi_family,-0.068807,0.019679,4.782474e-04,-0.107392,-0.030222
borough_1,1.199265,0.089030,3.387379e-40,1.024698,1.373831
borough_2,-0.283139,0.028749,1.525495e-22,-0.339510,-0.226768
borough_3,0.190421,0.021127,3.481199e-19,0.148997,0.231845
borough_5,-0.204413,0.024058,3.027528e-17,-0.251585,-0.157242


**Число наблюдений**: No. Observations:	3000

**$R^2$ для $\ln P$**: 0.386

### Модель 3. Наклон может различаться для двух типов домов

$$
\begin{aligned}
& \ln(P)=\beta_0+\beta_1z+\beta_2M+G+\delta Mz+\varepsilon \\
& G=\alpha_1D_1+\alpha_2D_2+\alpha_3D_3+\alpha_5D_5.
\end{aligned}
$$

Для домов с одним жилым помещением эластичность условной медианы цены по площади равна $\beta_1$, для домов с двумя-тремя помещениями — $\beta_1+\delta$.

In [185]:
# Площадь, тип дома и районные индикаторы.
selected_data['multi_log_area150'] = selected_data['multi_family'] * selected_data['log_area150']
features_3 = ['log_area150', 'multi_family', 'multi_log_area150', *dummy_columns]
X_3 = sm.add_constant(selected_data[features_3])
res_3 = sm.OLS(y, X_3).fit()
show_model(res_3)

,Коэффициент,Ст. ошибка,p-value,ДИ 95%: нижняя,ДИ 95%: верхняя
const,6.843298,0.015468,0.000000e+00,6.812970,6.873626
log_area150,0.756327,0.031119,3.098668e-119,0.695311,0.817343
multi_family,-0.045505,0.021196,3.188562e-02,-0.087065,-0.003944
multi_log_area150,-0.144136,0.049099,3.353892e-03,-0.240406,-0.047866
borough_1,1.194399,0.088933,5.640611e-40,1.020024,1.368774
borough_2,-0.279119,0.028746,5.747757e-22,-0.335482,-0.222755
borough_3,0.194134,0.021138,7.528851e-20,0.152688,0.235580
borough_5,-0.202995,0.024032,4.602073e-17,-0.250116,-0.155874


**Число наблюдений**: No. Observations:	3000

**$R^2$ для $\ln P$**: 0.388

## 3. Сравните цены

Для каждого прогноза подготовьте значения признаков, включая z и, в модели 3, Mz. Получите прогнозы логарифма цены через predict или подстановкой в уравнение модели. Возьмите экспоненту от каждого прогноза и посчитайте, на сколько процентов прогноз цены отличается от базового. В ноутбуке укажите значения признаков и покажите расчёт. Округляйте до двух знаков после запятой только итоговые проценты.

1. Выберите исходную площадь S. По модели 2 сравните прогноз для площади 1,10S с прогнозом для S, сохраняя тип дома и район.
2. Выберите одну площадь. По каждой из моделей 1 и 2 сравните прогноз для M = 1 с прогнозом для M = 0. Возьмите M = 0 за базу. Для модели 2 используйте один район в обоих прогнозах.
3. По модели 3 повторите сравнение типов домов при 150 м² и при 200 м². Используйте один район для всех прогнозов.
4. Как учёт района меняет оценку разницы между типами домов в моделях 1 и 2? Как эта разница меняется с площадью в модели 3?

In [186]:
def make_X(area, multi_family, borough=4):
    new = pd.DataFrame({'area': np.atleast_1d(area),
                        'multi_family': np.atleast_1d(multi_family),
                        'borough': np.atleast_1d(borough)})
    new['log_area150'] = np.log(new['area'] / 150)
    new['multi_log_area150'] = new['multi_family'] * new['log_area150']
    
    # Queens — база: для него все четыре индикатора равны нулю.
    for j in [1, 2, 3, 5]:
        new[f'borough_{j}'] = (new['borough'] == j).astype(float)
    
    return sm.add_constant(new[features_3], has_constant='add')[X_3.columns]


In [187]:
synthetic_S = 150

### 1

In [188]:
# 3.1. Увеличение площади на 10%, модель 2
# Подготовьте отдельную строку с признаками для каждого прогноза.
X_comp_1 = make_X(area=[synthetic_S, synthetic_S * 1.1], multi_family=[0, 0], borough=[4, 4])
X_comp_1 = X_comp_1[X_2.columns]

predict_1 = res_2.predict(X_comp_1).to_frame(
    name='Прогноз ln(P)'
)

predict_1['Прогноз цены, тыс. долл. США'] = np.exp(
    predict_1['Прогноз ln(P)']
)

prices = predict_1['Прогноз цены, тыс. долл. США']
change_pct = (prices.iloc[1] / prices.iloc[0] - 1) * 100


display(X_comp_1)
display(predict_1)
print(f'При увеличении площади на 10% прогноз цены изменился на {change_pct:.2f}%.')

,const,log_area150,multi_family,borough_1,borough_2,borough_3,borough_5
0,1.0,0.00000,0,0.0,0.0,0.0,0.0
1,1.0,0.09531,0,0.0,0.0,0.0,0.0


,Прогноз ln(P),"Прогноз цены, тыс. долл. США"
0,6.840392,934.855205
1,6.907262,999.507068


При увеличении площади на 10% прогноз цены изменился на 6.92%.


### 2

In [189]:
# 3.2. Сравнение типов домов, модели 1 и 2
X_comp_2 = make_X(area=[synthetic_S, synthetic_S], multi_family=[0, 1], borough=[4, 4])
X_comp_21 = X_comp_2[X_1.columns]

# Модель 1
predict_21 = res_1.predict(X_comp_21).to_frame(
    name='Прогноз ln(P)'
)
predict_21['Прогноз цены, тыс. долл. США'] = np.exp(
    predict_21['Прогноз ln(P)']
)
prices = predict_21['Прогноз цены, тыс. долл. США']
change_pct = (prices.iloc[1] / prices.iloc[0] - 1) * 100

# Модель 2
X_comp_22 = X_comp_2[X_2.columns]
predict_22 = res_2.predict(X_comp_22).to_frame(
    name='Прогноз ln(P)'
)
predict_22['Прогноз цены, тыс. долл. США'] = np.exp(
    predict_22['Прогноз ln(P)']
)
prices = predict_22['Прогноз цены, тыс. долл. США']
change_pct = (prices.iloc[1] / prices.iloc[0] - 1) * 100



# Сравнение
rows = []

for name, prediction in [
    ('Модель 1', predict_21),
    ('Модель 2', predict_22),
]:
    log_prices = prediction['Прогноз ln(P)']
    prices = prediction['Прогноз цены, тыс. долл. США']

    rows.append({
        'Модель': name,
        'ln(P), M=0': log_prices.iloc[0],
        'ln(P), M=1': log_prices.iloc[1],
        'Цена M=0, тыс. $': prices.iloc[0],
        'Цена M=1, тыс. $': prices.iloc[1],
        'Разница, %': (prices.iloc[1] / prices.iloc[0] - 1) * 100,
    })

comparison_models = pd.DataFrame(rows).set_index('Модель')

display(X_comp_2)
display(comparison_models.round({'Разница, %': 2}))

,const,log_area150,multi_family,multi_log_area150,borough_1,borough_2,borough_3,borough_5
0,1.0,0.0,0,0.0,0.0,0.0,0.0,0.0
1,1.0,0.0,1,0.0,0.0,0.0,0.0,0.0


,"ln(P), M=0","ln(P), M=1","Цена M=0, тыс. $","Цена M=1, тыс. $","Разница, %"
Модель,,,,,
Модель 1,6.826534,6.759944,921.989770,862.593825,-6.44
Модель 2,6.840392,6.771585,934.855205,872.694002,-6.65


### 3

In [190]:
# 3.3. Сравнение типов домов при 150 м² и 200 м², модель 3
X_comp_3 = make_X(area=[150, 150, 200, 200], multi_family=[0, 1, 0, 1], borough=[4, 4, 4, 4])
X_comp_3 = X_comp_3[X_3.columns]

predict_3 = res_3.predict(X_comp_3).to_frame(
    name='Прогноз ln(P)'
)

predict_3['Прогноз цены, тыс. долл. США'] = np.exp(
    predict_3['Прогноз ln(P)']
)

prices = predict_3['Прогноз цены, тыс. долл. США']

change_150 = (prices.iloc[1] / prices.iloc[0] - 1) * 100
change_200 = (prices.iloc[3] / prices.iloc[2] - 1) * 100


display(X_comp_3)
display(predict_3)

print(f'Разница типов при 150 м²: {change_150:.2f}%')
print(f'Разница типов при 200 м²: {change_200:.2f}%')


,const,log_area150,multi_family,multi_log_area150,borough_1,borough_2,borough_3,borough_5
0,1.0,0.000000,0,0.000000,0.0,0.0,0.0,0.0
1,1.0,0.000000,1,0.000000,0.0,0.0,0.0,0.0
2,1.0,0.287682,0,0.000000,0.0,0.0,0.0,0.0
3,1.0,0.287682,1,0.287682,0.0,0.0,0.0,0.0


,Прогноз ln(P),"Прогноз цены, тыс. долл. США"
0,6.843298,937.576378
1,6.797794,895.868400
2,7.060880,1165.470220
3,6.973910,1068.391896


Разница типов при 150 м²: -4.45%
Разница типов при 200 м²: -8.33%


**Сравнение результатов (пункт 3.4)**



### 4

**Как учёт района меняет оценку разницы между типами домов в моделях 1 и 2?**

В модели 1 при одинаковой площади прогноз цены домов с двумя-тремя жилыми помещениями ниже прогноза для домов с одним помещением на 6,44%. После учёта района в модели 2 разница составляет −6,65%. Учёт района мало изменяет оценку: разрыв увеличивается примерно на 0,21% в пользу домов с одним помещением.


**Как эта разница меняется с площадью в модели 3?**

При одинаковом районе прогноз цены домов с двумя-тремя помещениями ниже прогноза для домов с одним помещением на 4,45% при площади 150 м² и на 8,33% при площади 200 м². Следовательно, с увеличением площади разрыв увеличивается в пользу домов с одним помещением. Это согласуется с отрицательным коэффициентом у $Mz$ в модели 3.

## 4. Проверьте равенство наклонов

По модели 3 проверьте, одинаков ли наклон зависимости ln(P) от z для двух типов домов.

1. Перед расчётами сформулируйте нулевую и альтернативную гипотезы через коэффициент δ при Mz.
2. Проведите двусторонний t-тест для δ или F-тест для сравнения моделей 2 и 3.
3. Укажите значение статистики теста, её распределение при H₀ с числом степеней свободы и p-value.
4. Примите решение при уровне значимости 5%. Скажите, дают ли данные основания считать наклоны разными.

- $H_0: \delta = 0$ — наклоны зависимости логарифма цены от
  логарифма относительной площади одинаковы для двух типов домов.
- $H_1: \delta \ne 0$ — наклоны различаются.

Зададим уровень значимости $\alpha = 0{,}05$
(соответствующий 95%-му доверительному интервалу).

Если p-value < 0,05, отвергаем $H_0$; иначе не отвергаем.

In [191]:
ALPHA = 0.05

slope_test = res_3.t_test('multi_log_area150 = 0')
slope_test

<class 'statsmodels.stats.contrast.ContrastResults'>
                             Test for Constraints                             
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
c0            -0.1441      0.049     -2.936      0.003      -0.240      -0.048

**Результаты теста и вывод о наклонах**

- Выбранный тест: двусторонний t-тест
- Значение статистики: -2.9356
- Распределение статистики при $H_0$: распределение Стьюдента
- Степени свободы: 2992
- p-value: 0.003
- Решение при уровне значимости 5% и вывод о наклонах: Отвергаем $H_0$ (так как $p-value ≈ 0,003 < 0.05$)


## 5. Оцените медиану цены и постройте интервалы

1. Выпишите из таблицы target в ноутбуке площадь, тип и район дома для прогноза.
2. По модели 3 получите прогноз логарифма цены и 95%-й доверительный интервал для среднего логарифма. Возьмите экспоненту от прогноза и от обеих границ. Так вы получите оценку медианы цены и доверительный интервал для неё.
3. Постройте 95%-й прогнозный интервал для цены одного нового дома с теми же характеристиками. Сначала найдите границы на шкале логарифма, затем возьмите экспоненту от каждой. Объясните, почему прогнозный интервал шире доверительного.

Запишите оценку медианы и границы обоих интервалов в тысячах долларов.

### 1

In [192]:
target

,area_m2,multi_family,borough
0,160,0,5


In [201]:
X_target = make_X(
    area=target['area_m2'],
    multi_family=target['multi_family'],
    borough=target['borough'],
)

X_target

,const,log_area150,multi_family,multi_log_area150,borough_1,borough_2,borough_3,borough_5
0,1.0,0.064539,0,0.0,0.0,0.0,0.0,1.0


### 2

In [209]:
# В statsmodels границы двух интервалов можно получить через get_prediction().summary_frame().
# Если передаёте матрицу признаков, включите константу и сохраните порядок колонок модели 3.
# При вызове sm.add_constant для одной строки укажите has_constant='add'.
pred = res_3.get_prediction(X_target)
prediction_log = pred.summary_frame(alpha=ALPHA)
prediction_log


,mean,mean_se,mean_ci_lower,mean_ci_upper,obs_ci_lower,obs_ci_upper
0,6.689115,0.021035,6.647871,6.73036,5.786427,7.591804


Оценка медианы цены и доверительный интервал для неё

In [211]:
prediction_price = np.exp(
    prediction_log[[
        'mean',
        'mean_ci_lower', 'mean_ci_upper',
        'obs_ci_lower', 'obs_ci_upper',
    ]]
).rename(columns={
    'mean': 'Медиана цены',
    'mean_ci_lower': 'ДИ 95%: нижняя',
    'mean_ci_upper': 'ДИ 95%: верхняя',
    'obs_ci_lower': 'ПИ 95%: нижняя',
    'obs_ci_upper': 'ПИ 95%: верхняя',
})

prediction_price

,Медиана цены,ДИ 95%: нижняя,ДИ 95%: верхняя,ПИ 95%: нижняя,ПИ 95%: верхняя
0,803.610911,771.140671,837.448368,325.846578,1981.885158


**Характеристики дома и объяснение разницы между интервалами**



Характеристики дома из target:
- площадь — 160 м²;
- тип — одно жилое помещение $M=0$;
- район — Staten Island (5).

Оценка медианы цены составляет 803,61 тыс. долларов США. 

Доверительный интервал для медианы — [771,14; 837,45] тыс. долл. США, а 95%-й прогнозный интервал для цены одного нового дома — [325,85; 1981,89] тыс. долл. США.

**Объясните, почему прогнозный интервал шире доверительного.**

Доверительный интервал учитывает неопределённость оценки предсказываемого среднего. Прогнозный интервал дополнительно учитывает случайное отклонение нового наблюдения от этого среднего, поэтому он шире.

## 6. Подведите итог и заполните лист с результатами

1. Напишите в ноутбуке общий вывод в 5–7 предложениях. Опишите, как учёт района изменил оценку разницы в ценах двух типов домов и совпало ли это с вашими ожиданиями. Выберите модель для описания этой разницы, опираясь на численные результаты заданий 3 и 4. Назовите одно ограничение анализа.
2. Заполните лист с результатами модели 3 на последней странице PDF.

**Общий вывод**

